# K-Means Clustering

## ¿Cuál es el objetivo de K-Means?

**K-Means** es un algoritmo de **clustering** (agrupamiento) cuyo objetivo es dividir un conjunto de $n$ datos en **$K$ grupos (clusters)**, de forma que los puntos dentro de un mismo cluster sean lo más **similares** entre sí y lo más **diferentes** posible de los puntos en otros clusters.

Al igual que PCA, K-Means es un algoritmo de **aprendizaje no supervisado**: no necesita etiquetas $y$, solo trabaja con las features $\mathbf{X}$. Es el algoritmo quien "descubre" la estructura de grupos en los datos.

La idea central es:

1. Definir $K$ **centroides** (centros de cada cluster).
2. Asignar cada punto al centroide más cercano.
3. Recalcular los centroides como la **media** de los puntos asignados.
4. Repetir hasta que los centroides **no cambien** (convergencia).

Formalmente, K-Means minimiza la **inercia** (Within-Cluster Sum of Squares, WCSS):

$$
J = \sum_{k=1}^{K} \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2
$$

Donde:

- $C_k$: conjunto de puntos asignados al cluster $k$.
- $\boldsymbol{\mu}_k$: centroide del cluster $k$ (media de sus puntos).
- $\|\cdot\|^2$: distancia euclidiana al cuadrado.

**¿Para qué sirve?**
- **Segmentación de clientes**: agrupar usuarios con comportamientos similares.
- **Compresión de imágenes**: reducir colores agrupando píxeles similares.
- **Detección de anomalías**: puntos lejanos a todos los centroides son sospechosos.
- **Preprocesamiento**: crear features de pertenencia a clusters para otros modelos.

**Supongamos un conjunto de datos simples**

Consideremos puntos en 2D que forman tres grupos naturales:

| $x_1$ | $x_2$ | Grupo visual |
|-------|-------|-------------|
| 1.0   | 1.0   | A           |
| 1.5   | 1.5   | A           |
| 1.2   | 2.0   | A           |
| 5.0   | 5.0   | B           |
| 5.5   | 4.5   | B           |
| 5.0   | 5.5   | B           |
| 9.0   | 1.0   | C           |
| 9.5   | 1.5   | C           |
| 9.0   | 2.0   | C           |

A simple vista podemos ver tres grupos, pero el algoritmo **no sabe** que existen. K-Means los descubrirá automáticamente si le indicamos $K = 3$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Datos con 3 grupos naturales
X = np.array([
    [1.0, 1.0], [1.5, 1.5], [1.2, 2.0],   # Grupo A
    [5.0, 5.0], [5.5, 4.5], [5.0, 5.5],   # Grupo B
    [9.0, 1.0], [9.5, 1.5], [9.0, 2.0],   # Grupo C
])

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], color='gray', s=100, edgecolors='black')

for i, (x, y) in enumerate(X):
    plt.annotate(f'  ({x:.1f}, {y:.1f})', (x, y), fontsize=9)

plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Datos sin etiquetar: ¿puedes ver los 3 grupos?')
plt.grid(True, alpha=0.3)
plt.show()

## ¿Cómo encuentra K-Means los clusters?

K-Means utiliza un proceso **iterativo** de dos pasos que se repite hasta convergencia:

1. **Asignar** cada punto al centroide más cercano (usando distancia euclidiana).
2. **Actualizar** cada centroide como la media de los puntos asignados.

Este proceso se conoce como el **algoritmo de Lloyd** y está garantizado a converger, aunque no necesariamente al óptimo global (puede quedar atrapado en mínimos locales dependiendo de la inicialización).

## Demostración Matemática

### Paso 1: Inicializar centroides

Seleccionamos $K$ puntos del dataset al azar como centroides iniciales:

$$
\boldsymbol{\mu}_1^{(0)}, \ \boldsymbol{\mu}_2^{(0)}, \ \dots, \ \boldsymbol{\mu}_K^{(0)}
$$

El superíndice $(0)$ indica que son los centroides en la iteración 0.

**Nota importante:** la inicialización es crucial. Distintas inicializaciones pueden llevar a distintos resultados. En la práctica se usa **K-Means++**, que selecciona centroides iniciales bien dispersos. Nuestra implementación usa selección aleatoria por simplicidad.

### Paso 2: Asignar puntos a clusters (paso E)

Cada punto $\mathbf{x}_i$ se asigna al cluster cuyo centroide está más cerca:

$$
c_i = \arg\min_k \ \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2 = \arg\min_k \ \sum_{j=1}^{d} (x_{ij} - \mu_{kj})^2
$$

Donde:

- $c_i$: cluster asignado al punto $\mathbf{x}_i$.
- $d$: número de features.

Esto crea una **partición** del espacio: cada punto del espacio pertenece al cluster cuyo centroide es el más cercano. Esta partición se llama **diagrama de Voronoi**.

Con centroides fijos, este paso **minimiza** la función de costo $J$ respecto a las asignaciones $c_i$.

### Paso 3: Actualizar centroides (paso M)

Una vez asignados los puntos, recalculamos cada centroide como la **media** de los puntos en su cluster:

$$
\boldsymbol{\mu}_k^{(t+1)} = \frac{1}{|C_k|} \sum_{\mathbf{x}_i \in C_k} \mathbf{x}_i
$$

Donde $|C_k|$ es el número de puntos en el cluster $k$.

**¿Por qué la media?** Porque la media es el punto que minimiza la suma de distancias cuadráticas a todos los puntos del grupo:

$$
\boldsymbol{\mu}_k^* = \arg\min_{\boldsymbol{\mu}} \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}\|^2
$$

Derivando respecto a $\boldsymbol{\mu}$ e igualando a cero:

$$
\frac{\partial}{\partial \boldsymbol{\mu}} \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}\|^2 = -2 \sum_{\mathbf{x}_i \in C_k} (\mathbf{x}_i - \boldsymbol{\mu}) = 0
$$

$$
\implies \boldsymbol{\mu} = \frac{1}{|C_k|} \sum_{\mathbf{x}_i \in C_k} \mathbf{x}_i
$$

Con asignaciones fijas, este paso **minimiza** $J$ respecto a los centroides $\boldsymbol{\mu}_k$.

### Paso 4: Convergencia

Repetimos los pasos 2 y 3 hasta que los centroides **no cambien** entre iteraciones:

$$
\sum_{k=1}^{K} \|\boldsymbol{\mu}_k^{(t+1)} - \boldsymbol{\mu}_k^{(t)}\| = 0
$$

**¿Por qué converge?** Cada paso (asignación y actualización) solo puede **reducir o mantener** la función de costo $J$:

- El paso de **asignación** reduce $J$ porque cada punto se mueve al centroide más cercano.
- El paso de **actualización** reduce $J$ porque la media minimiza las distancias cuadráticas.

Como $J \geq 0$ y es monótonamente decreciente, el algoritmo **siempre converge**. Sin embargo, puede converger a un **mínimo local**, no necesariamente al global.

### La importancia de $K$

El valor de $K$ (número de clusters) es el hiperparámetro principal:

- **$K$ muy pequeño**: fusiona clusters que deberían estar separados.
- **$K$ muy grande**: divide clusters naturales en sub-clusters artificiales.

Un método popular para elegir $K$ es el **Método del Codo (Elbow Method)**: se grafica la inercia ($J$) para distintos valores de $K$ y se busca el punto donde la curva deja de decrecer significativamente (el "codo").

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

# Datos de ejemplo
X = np.array([
    [1.0, 1.0], [1.5, 1.5], [1.2, 2.0],
    [5.0, 5.0], [5.5, 4.5], [5.0, 5.5],
    [9.0, 1.0], [9.5, 1.5], [9.0, 2.0],
])

K = 3

# Inicialización: seleccionar 3 puntos aleatorios como centroides
random_idxs = np.random.choice(len(X), K, replace=False)
centroids = X[random_idxs].copy()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for it in range(6):
    ax = axes[it // 3][it % 3]

    # Paso E: asignar puntos
    labels = np.array([np.argmin([np.sqrt(np.sum((x - c)**2)) for c in centroids]) for x in X])

    # Graficar
    for k in range(K):
        mask = labels == k
        ax.scatter(X[mask, 0], X[mask, 1], color=colors[k], s=80, edgecolors='black', alpha=0.7)
    ax.scatter(centroids[:, 0], centroids[:, 1], color='black', marker='X', s=200, zorder=5,
               edgecolors='white', linewidths=2)

    # Dibujar líneas a centroides
    for i, x in enumerate(X):
        ax.plot([x[0], centroids[labels[i], 0]], [x[1], centroids[labels[i], 1]],
                color=colors[labels[i]], alpha=0.3, linestyle='--')

    # Inercia
    inertia = sum(np.sum((X[labels == k] - centroids[k])**2) for k in range(K))
    ax.set_title(f'Iteración {it + 1} (J = {inertia:.2f})', fontsize=11)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.grid(True, alpha=0.3)

    # Paso M: actualizar centroides
    new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])

    if np.allclose(centroids, new_centroids):
        # Rellenar los ejes restantes
        for j in range(it + 1, 6):
            ax_empty = axes[j // 3][j % 3]
            ax_empty.set_visible(False)
        ax.set_title(f'Iteración {it + 1} - ¡Convergió! (J = {inertia:.2f})', fontsize=11)
        break
    centroids = new_centroids

plt.suptitle('K-Means paso a paso', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Ejemplo numérico

Apliquemos K-Means con $K = 3$ a nuestros 9 puntos.

**Inicialización:** Supongamos que los centroides iniciales son:

$$
\boldsymbol{\mu}_1^{(0)} = (1.2, 2.0), \quad \boldsymbol{\mu}_2^{(0)} = (5.0, 5.5), \quad \boldsymbol{\mu}_3^{(0)} = (9.0, 1.0)
$$

**Iteración 1 — Asignación:**

Para cada punto, calculamos la distancia a cada centroide y asignamos al más cercano:

| Punto | $d(\boldsymbol{\mu}_1)$ | $d(\boldsymbol{\mu}_2)$ | $d(\boldsymbol{\mu}_3)$ | Cluster |
|-------|-----------|-----------|-----------|--------|
| (1.0, 1.0) | 1.02 | 5.70 | 8.00 | 1 |
| (1.5, 1.5) | 0.58 | 5.15 | 7.50 | 1 |
| (1.2, 2.0) | 0.00 | 5.22 | 7.87 | 1 |
| (5.0, 5.0) | 4.82 | 0.50 | 5.66 | 2 |
| (5.5, 4.5) | 4.95 | 1.12 | 4.95 | 2 |
| (5.0, 5.5) | 5.22 | 0.00 | 5.70 | 2 |
| (9.0, 1.0) | 7.87 | 5.70 | 0.00 | 3 |
| (9.5, 1.5) | 8.30 | 5.41 | 0.71 | 3 |
| (9.0, 2.0) | 7.80 | 5.22 | 1.00 | 3 |

**Iteración 1 — Actualización de centroides:**

$$
\boldsymbol{\mu}_1^{(1)} = \frac{1}{3}[(1.0, 1.0) + (1.5, 1.5) + (1.2, 2.0)] = (1.233, 1.500)
$$

$$
\boldsymbol{\mu}_2^{(1)} = \frac{1}{3}[(5.0, 5.0) + (5.5, 4.5) + (5.0, 5.5)] = (5.167, 5.000)
$$

$$
\boldsymbol{\mu}_3^{(1)} = \frac{1}{3}[(9.0, 1.0) + (9.5, 1.5) + (9.0, 2.0)] = (9.167, 1.500)
$$

En la **Iteración 2**, al reasignar los puntos con los nuevos centroides, las asignaciones no cambian → **el algoritmo ha convergido**.

La inercia final:

$$
J = \sum_{k=1}^{3} \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2 \approx 2.33
$$

In [ ]:
import numpy as np

X = np.array([
    [1.0, 1.0], [1.5, 1.5], [1.2, 2.0],
    [5.0, 5.0], [5.5, 4.5], [5.0, 5.5],
    [9.0, 1.0], [9.5, 1.5], [9.0, 2.0],
])

# Centroides iniciales (puntos 2, 5, 6)
centroids = np.array([[1.2, 2.0], [5.0, 5.5], [9.0, 1.0]])

print("=== Ejemplo numérico: K-Means paso a paso ===")
print(f"Centroides iniciales:")
for k, c in enumerate(centroids):
    print(f"  μ_{k+1} = ({c[0]:.3f}, {c[1]:.3f})")

for it in range(5):
    print(f"\n--- Iteración {it + 1} ---")

    # Asignar
    labels = []
    print("\nAsignación:")
    for i, x in enumerate(X):
        dists = [np.sqrt(np.sum((x - c)**2)) for c in centroids]
        label = np.argmin(dists)
        labels.append(label)
        print(f"  ({x[0]:.1f}, {x[1]:.1f}) → dists=[{dists[0]:.2f}, {dists[1]:.2f}, {dists[2]:.2f}] → Cluster {label + 1}")
    labels = np.array(labels)

    # Actualizar
    new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(3)])
    print("\nNuevos centroides:")
    for k, c in enumerate(new_centroids):
        print(f"  μ_{k+1} = ({c[0]:.3f}, {c[1]:.3f})")

    # Inercia
    inertia = sum(np.sum((X[labels == k] - new_centroids[k])**2) for k in range(3))
    print(f"Inercia J = {inertia:.4f}")

    # Convergencia
    if np.allclose(centroids, new_centroids):
        print("\n¡Convergió! Los centroides no cambiaron.")
        break
    centroids = new_centroids

## Implementación con Código

### Funciones

In [ ]:
import numpy as np

def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

def kmeans_fit(X, K, max_iters=100, random_state=42):
    """Ejecuta K-Means y retorna centroides y etiquetas"""
    np.random.seed(random_state)
    n_samples, n_features = X.shape

    # Inicializar centroides aleatorios
    random_idxs = np.random.choice(n_samples, K, replace=False)
    centroids = X[random_idxs].copy()

    for _ in range(max_iters):
        # Asignar cada punto al centroide más cercano
        labels = np.array([
            np.argmin([euclidean_distance(x, c) for c in centroids])
            for x in X
        ])

        # Actualizar centroides
        new_centroids = np.array([
            X[labels == k].mean(axis=0) for k in range(K)
        ])

        # Verificar convergencia
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids

    return centroids, labels

def kmeans_inertia(X, centroids, labels):
    """Calcula la inercia (WCSS)"""
    return sum(
        np.sum((X[labels == k] - centroids[k]) ** 2)
        for k in range(len(centroids))
    )

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Generar datos
X, y_true = make_blobs(centers=3, n_samples=500, n_features=2, shuffle=True, random_state=40)
print(f"Shape: {X.shape}")

# Aplicar K-Means
K = 3
centroids, labels = kmeans_fit(X, K)

# Graficar
plt.figure(figsize=(8, 6))
for k in range(K):
    mask = labels == k
    plt.scatter(X[mask, 0], X[mask, 1], s=40, alpha=0.6)
plt.scatter(centroids[:, 0], centroids[:, 1], color='black', marker='X', s=200,
            edgecolors='white', linewidths=2, label='Centroides')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title(f'K-Means con K={K}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

inertia = kmeans_inertia(X, centroids, labels)
print(f"Inercia (WCSS): {inertia:.2f}")

### Clase

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)


def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))


class KMeans:
    def __init__(self, K=5, max_iters=100, plot_steps=False):
        self.K = K
        self.max_iters = max_iters
        self.plot_steps = plot_steps

        # Lista de índices de muestras para cada cluster
        self.clusters = [[] for _ in range(self.K)]
        # Centroides de cada cluster
        self.centroids = []

    def predict(self, X):
        """Ejecuta K-Means y retorna las etiquetas de cluster"""
        self.X = X
        self.n_samples, self.n_features = X.shape

        # Inicializar centroides aleatorios
        random_sample_idxs = np.random.choice(self.n_samples, self.K, replace=False)
        self.centroids = [self.X[idx] for idx in random_sample_idxs]

        # Optimizar clusters
        for _ in range(self.max_iters):
            # Asignar muestras al centroide más cercano
            self.clusters = self._create_clusters(self.centroids)

            if self.plot_steps:
                self.plot()

            # Calcular nuevos centroides
            centroids_old = self.centroids
            self.centroids = self._get_centroids(self.clusters)

            # Verificar convergencia
            if self._is_converged(centroids_old, self.centroids):
                break

            if self.plot_steps:
                self.plot()

        return self._get_cluster_labels(self.clusters)

    def _get_cluster_labels(self, clusters):
        """Asigna a cada muestra la etiqueta de su cluster"""
        labels = np.empty(self.n_samples)
        for cluster_idx, cluster in enumerate(clusters):
            for sample_index in cluster:
                labels[sample_index] = cluster_idx
        return labels

    def _create_clusters(self, centroids):
        """Asigna muestras al centroide más cercano"""
        clusters = [[] for _ in range(self.K)]
        for idx, sample in enumerate(self.X):
            centroid_idx = self._closest_centroid(sample, centroids)
            clusters[centroid_idx].append(idx)
        return clusters

    def _closest_centroid(self, sample, centroids):
        """Encuentra el centroide más cercano a una muestra"""
        distances = [euclidean_distance(sample, point) for point in centroids]
        closest_index = np.argmin(distances)
        return closest_index

    def _get_centroids(self, clusters):
        """Calcula nuevos centroides como la media de cada cluster"""
        centroids = np.zeros((self.K, self.n_features))
        for cluster_idx, cluster in enumerate(clusters):
            cluster_mean = np.mean(self.X[cluster], axis=0)
            centroids[cluster_idx] = cluster_mean
        return centroids

    def _is_converged(self, centroids_old, centroids):
        """Verifica si los centroides cambiaron"""
        distances = [
            euclidean_distance(centroids_old[i], centroids[i]) for i in range(self.K)
        ]
        return sum(distances) == 0

    def inertia(self):
        """Calcula la inercia (WCSS)"""
        total = 0
        for cluster_idx, cluster in enumerate(self.clusters):
            for sample_idx in cluster:
                total += np.sum((self.X[sample_idx] - self.centroids[cluster_idx]) ** 2)
        return total

    def plot(self):
        """Grafica el estado actual de los clusters"""
        fig, ax = plt.subplots(figsize=(10, 7))

        for i, index in enumerate(self.clusters):
            point = self.X[index].T
            ax.scatter(*point, alpha=0.6)

        for point in self.centroids:
            ax.scatter(*point, marker="X", color="black", s=200,
                       edgecolors='white', linewidths=2)

        ax.set_xlabel('$x_1$')
        ax.set_ylabel('$x_2$')
        ax.set_title(f'K-Means (K={self.K})')
        ax.grid(True, alpha=0.3)
        plt.show()

In [ ]:
from sklearn.datasets import make_blobs

# Generar datos
X, y_true = make_blobs(
    centers=3, n_samples=500, n_features=2, shuffle=True, random_state=40
)

print(f"Shape: {X.shape}")
print(f"Clusters reales: {len(np.unique(y_true))}")

# Aplicar K-Means
k = KMeans(K=3, max_iters=150, plot_steps=False)
y_pred = k.predict(X)

# Resultado final
k.plot()
print(f"Inercia final: {k.inertia():.2f}")

In [ ]:
# Método del Codo para elegir K
inertias = []
K_range = range(1, 10)

for K_val in K_range:
    km = KMeans(K=K_val, max_iters=150)
    km.predict(X)
    inertias.append(km.inertia())

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-', linewidth=2)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.5, label='K=3 (codo)')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Inercia (WCSS)')
plt.title('Método del Codo')
plt.xticks(list(K_range))
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Visualizar con plot_steps=True para ver la evolución
print("Evolución de K-Means paso a paso:")
k_visual = KMeans(K=3, max_iters=150, plot_steps=True)
y_pred_visual = k_visual.predict(X)